# NB52: Spark + Cassandra

Persisting streaming sensor data to Cassandra.

## 1. Environment Setup

This cell installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and necessary Python libraries (`pyspark`, `kafka-python`, `redis`, `pymongo`, `elasticsearch`, `cassandra-driver`, `minio`). It also sets environment variables for Java and Spark.

In [ ]:
# Install Dependencies
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip install -q findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio "numpy<2.0.0"

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

This cell starts the required distributed services in the background:
- **Kafka & Zookeeper**: Event streaming platform.
- **Cassandra**: Wide-column store.

In [ ]:
# Start Kafka
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start Cassandra
!wget -q https://archive.apache.org/dist/cassandra/4.1.3/apache-cassandra-4.1.3-bin.tar.gz
!tar xf apache-cassandra-4.1.3-bin.tar.gz
!apache-cassandra-4.1.3/bin/cassandra -R > cassandra.log 2>&1 &

import time, socket, os
def wait_for_port(port, host='localhost', timeout=120):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port} to start.")
                # Dump logs for debugging
                if os.path.exists('minio.log'):
                    print('--- MINIO LOG ---')
                    print(open('minio.log').read())
                if os.path.exists('es.log'):
                    print('--- ES LOG ---')
                    print(open('es.log').read())
                if os.path.exists('cassandra.log'):
                    print('--- CASSANDRA LOG ---')
                    print(open('cassandra.log').read())
                raise Exception(f"Service at {host}:{port} failed to start.")
            time.sleep(2)

# Wait for services
wait_for_port(9092) # Kafka
wait_for_port(9042) # Cassandra
time.sleep(10) # Extra buffer for Cassandra


## 3. Create Kafka Topic

Creates a topic named `input-topic` with 1 partition and replication factor 1.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer

Simulates sensor data (`id`, `temp`) and sends JSON messages to Kafka.

In [ ]:
from kafka import KafkaProducer
import json, time, random
print("Starting Sensor Data Producer...")
producer = KafkaProducer(bootstrap_servers='localhost:9092', value_serializer=lambda v: json.dumps(v).encode('utf-8'))
print("Sending 100 sensor readings...")
for _ in range(100):
    data = {'id': f's{random.randint(1,5)}', 'temp': random.uniform(20.0, 30.0)}
    producer.send('input-topic', data)
producer.flush()
print("Producer finished.")

## 5. Spark -> Cassandra

Initialize Cassandra keyspace/table and use Spark `foreachBatch` to insert data.

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
from cassandra.cluster import Cluster
import json

# Init Cassandra Schema
cluster = Cluster(['127.0.0.1'])
session = cluster.connect()
session.execute("CREATE KEYSPACE IF NOT EXISTS sensors WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}")
session.execute("CREATE TABLE IF NOT EXISTS sensors.data (id text PRIMARY KEY, temp float)")
session.shutdown()

spark = SparkSession.builder.appName("Cassandra").getOrCreate()

def process_batch(df, epoch_id):
    rows = df.collect()
    cluster_local = Cluster(['127.0.0.1'])
    session_local = cluster_local.connect('sensors')
    for row in rows:
        val = json.loads(row.value)
        session_local.execute(f"INSERT INTO data (id, temp) VALUES ('{val['id']}', {val['temp']})")
    session_local.shutdown()
    print(f"Batch {epoch_id} persisted.")

df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").option("startingOffsets", "earliest").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Query Cassandra to verify data storage.

In [ ]:
from cassandra.cluster import Cluster
cluster = Cluster(['127.0.0.1'])
session = cluster.connect('sensors')
rows = session.execute("SELECT * FROM data LIMIT 10")
print("--- Data in Cassandra ---")
for row in rows: print(row)
session.shutdown()